Feature Engineering

In [1]:
import os
from pathlib import Path

def has_project_files(path: Path) -> bool:
    try:
        return (path / "config" / "config.yaml").is_file() and (path / "src").is_dir()
    except OSError:
        return False

def find_project_root(start: Path = Path.cwd()) -> Path:
    try:
        start = start.resolve()
    except OSError:
        start = Path.cwd()

    seeds = [start, Path.home() / "Documents" / "projects" / "Summarizer"]
    candidates = []
    for seed in seeds:
        try:
            seed = seed.resolve()
        except OSError:
            pass
        for candidate in [seed, *seed.parents]:
            if candidate not in candidates:
                candidates.append(candidate)
    for candidate in candidates:
        if has_project_files(candidate):
            return candidate

    for candidate in candidates:
        try:
            children = list(candidate.iterdir())
        except OSError:
            continue
        for child in children:
            try:
                if child.is_dir() and has_project_files(child):
                    return child
            except OSError:
                continue
    raise FileNotFoundError("Could not find project root containing config/config.yaml")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_ingestion_dir: Path
    tokenizer_name: Path

In [3]:
from src.Text_Summarizer.utils.common import read_yaml, create_directories
from src.Text_Summarizer.constants import *

In [4]:
class ConfigurationManager:
    def __init__(self, config_filepath: Path = CONFIG_FILE_PATH, params_filepath: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformer
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_ingestion_dir=Path(config.data_ingestion_dir),
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config




In [5]:
import os
from src.Text_Summarizer.logger import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


/Users/tyrion/Documents/projects/Summarizer/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

        target_encodings = self.tokenizer(example_batch['summary'], max_length=1024, truncation=True)

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_ingestion_dir)
        dataset_samsum = dataset_samsum.map(self.convert_examples_to_features, batched=True, batch_size=32)
        dataset_samsum.save_to_disk(os.path.join(self.config.root_dir, 'samsum_dataset'))



In [7]:
config = ConfigurationManager()
data_tranformation_config = config.get_data_transformation_config()
data_tranformation = DataTransformation(config=data_tranformation_config) 

data_tranformation.convert()

[2026-07-07 14:30:01,823: INFO: common]: YAML file: /Users/tyrion/Documents/projects/Summarizer/config/config.yaml loaded successfully.
[2026-07-07 14:30:01,825: INFO: common]: YAML file: /Users/tyrion/Documents/projects/Summarizer/params.yaml loaded successfully.
[2026-07-07 14:30:01,825: INFO: common]: Directory created at: artifacts
[2026-07-07 14:30:01,825: INFO: common]: Directory created at: artifacts/data_transformer
[2026-07-07 14:30:02,476: INFO: _client]: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[2026-07-07 14:30:02,747: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


[2026-07-07 14:30:02,748: WARNING: _http]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-07-07 14:30:02,774: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-07-07 14:30:03,038: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-07-07 14:30:03,064: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"
[2026-07-07 14:30:03,336: INFO: _client]: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Fou

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 379570.82 examples/s]
